In [1]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/spotify_songs.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10

In [2]:
num_data, dim = X_scaled.shape
np.random.seed(42)
sample_indices = np.random.choice(num_data, size=100, replace=False)
queries = X_scaled[sample_indices]

def calculate_recall(ground_truth, predictions):
    recalls = []
    for gt, pred in zip(ground_truth, predictions):
        intersection = len(set(gt).intersection(set(pred)))
        recalls.append(intersection/len(gt))
    return np.mean(recalls)

benchmark_results = []

# NN

In [3]:
start=time.time()
nn_model = NearestNeighbors(n_neighbors=k, metric='euclidean', algorithm='brute')
nn_model.fit(X_scaled)
nn_build_time = time.time()-start

start=time.time()
nn_distances, nn_indices = nn_model.kneighbors(queries)
nn_query_time = (time.time()-start)/len(queries)

benchmark_results.append({
    'Algoritma': 'NN',
    'Build Time (s)': f"{nn_build_time:.6f}",
    'Query Time (s)': f"{nn_query_time:.6f}",
    'Recall': '100.00%'
})

d:\Download\anaconda\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "d:\Download\anaconda\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "d:\Download\anaconda\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Download\anaconda\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~

# FAISS

In [4]:
X_faiss = X_scaled.astype('float32')
queries_faiss = queries.astype('float32')

start = time.time()
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(X_faiss)
faiss_build_time = time.time() - start

start = time.time()
_, faiss_indices = faiss_index.search(queries_faiss, k)
faiss_query_time = (time.time() - start) / len(queries)
faiss_recall = calculate_recall(nn_indices, faiss_indices)

benchmark_results.append({
    'Algoritma': 'FAISS',
    'Build Time (s)': f"{faiss_build_time:.6f}",
    'Query Time (s)': f"{faiss_query_time:.6f}",
    'Recall': f"{faiss_recall * 100:.2f}%"
})

# ANNOY

In [5]:
start = time.time()
annoy_index = AnnoyIndex(dim, metric='euclidean')
for i, vec in enumerate(X_scaled):
    annoy_index.add_item(i, vec)
annoy_index.build(n_trees=20)
annoy_build_time = time.time() - start

start = time.time()
annoy_indices = [annoy_index.get_nns_by_vector(q, k) for q in queries]
annoy_query_time = (time.time() - start) / len(queries)
annoy_recall = calculate_recall(nn_indices, annoy_indices)

benchmark_results.append({
    'Algoritma': 'ANNOY',
    'Build Time (s)': f"{annoy_build_time:.6f}",
    'Query Time (s)': f"{annoy_query_time:.6f}",
    'Recall': f"{annoy_recall * 100:.2f}%"
})

# HNSW

In [6]:
start = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=num_data, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled, np.arange(num_data))
hnsw_build_time = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
hnsw_indices, _ = hnsw_index.knn_query(queries, k=k)
hnsw_query_time = (time.time() - start) / len(queries)
hnsw_recall = calculate_recall(nn_indices, hnsw_indices)

benchmark_results.append({
    'Algoritma': 'HNSWLIB',
    'Build Time (s)': f"{hnsw_build_time:.6f}",
    'Query Time (s)': f"{hnsw_query_time:.6f}",
    'Recall': f"{hnsw_recall * 100:.2f}%"
})

In [7]:
df_results = pd.DataFrame(benchmark_results)
print("Hasil perbandingan ")
print(df_results.to_markdown(index=False))

Hasil perbandingan 
| Algoritma   |   Build Time (s) |   Query Time (s) | Recall   |
|:------------|-----------------:|-----------------:|:---------|
| NN          |         0.052833 |         0.00491  | 100.00%  |
| FAISS       |         0.042235 |         0.001475 | 99.20%   |
| ANNOY       |        12.3035   |         0.000137 | 94.60%   |
| HNSWLIB     |        29.7443   |         7.2e-05  | 98.40%   |
